# Phase 0.5 - Observability Estimators

Notebook này chạy bước tiếp theo sau Gate 2.5 và Phase 0.5 preflight: feature extraction + task-contrast observability estimator workflow.

Phạm vi khoa học:
- Chỉ chạy Phase 0.5 observability estimators.
- Không train decoder Phase 1.
- Không tạo claim privileged-transfer efficacy.
- Không được xem teacher survival draft là kết luận cuối nếu spatial/ICA controls hoặc permutation count final chưa hoàn tất.

Source of truth đã khóa:
- Gate 0: `/content/drive/MyDrive/eeg-ds004752/artifacts/gate0/20260417T102811097110Z`
- Gate 1: `/content/drive/MyDrive/eeg-ds004752/artifacts/gate1/20260418T153918409528Z`
- Gate 2: `/content/drive/MyDrive/eeg-ds004752/artifacts/gate2/20260418T160143330194Z`
- Gate 2.5: `/content/drive/MyDrive/eeg-ds004752/artifacts/prereg/20260418T161442014597Z`
- Phase 0.5 preflight: `/content/drive/MyDrive/eeg-ds004752/artifacts/phase05/20260418T163438037205Z`


In [ ]:
# ============================================================
# Cell 1: Mount Drive, clone/pull repo, install signal runtime
# ============================================================
# Mục tiêu:
# - Mount Google Drive.
# - Đảm bảo repo private đã clone tại /content/eeg-ds004752.
# - Pull code mới nhất có phase05_estimators CLI.
# - Cài signal extras: mne, numpy, scipy, scikit-learn nếu bootstrap hỗ trợ.
#
# Token GitHub không được in ra output.
# ============================================================

from pathlib import Path
from google.colab import drive
import base64
import getpass
import os
import subprocess

# Mount Drive.
drive.mount('/content/drive')

REPO_DIR = Path('/content/eeg-ds004752')
REPO_URL = 'https://github.com/BrianNguyen29/eeg-ds004752.git'


def run(cmd, cwd=None, check=True, capture_output=False):
    print('\n$', ' '.join(str(x) for x in cmd))
    return subprocess.run(cmd, cwd=cwd, check=check, text=True, capture_output=capture_output)


def git_auth_header():
    token = os.environ.get('GITHUB_TOKEN')
    if not token:
        token = getpass.getpass('Nhập GitHub token cho repo private nếu cần: ')
    basic = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    return f'Authorization: Basic {basic}'


if not REPO_DIR.exists():
    header = git_auth_header()
    run(['git', '-c', f'http.extraHeader={header}', 'clone', REPO_URL, str(REPO_DIR)])
else:
    print('Repo đã tồn tại:', REPO_DIR)

%cd /content/eeg-ds004752

pull = subprocess.run(['git', 'pull'], cwd=REPO_DIR)
if pull.returncode != 0:
    header = git_auth_header()
    run(['git', '-c', f'http.extraHeader={header}', 'pull'], cwd=REPO_DIR)

# Signal extras cần cho EDF reading và feature extraction.
run(['bash', '-lc', 'INSTALL_SIGNAL_EXTRAS=1 bash bootstrap/install_runtime.sh'], cwd=REPO_DIR)

run(['git', 'log', '--oneline', '-5'], cwd=REPO_DIR)


In [ ]:
# ============================================================
# Cell 2: Khai báo source of truth và đường dẫn output
# ============================================================
# Không dùng latest.txt cho các gate đã khóa, vì source of truth phải cố định.
# ============================================================

from pathlib import Path
import json
import hashlib

REPO_DIR = Path('/content/eeg-ds004752')
DRIVE_ROOT = Path('/content/drive/MyDrive/eeg-ds004752')
DATASET_ROOT = DRIVE_ROOT / 'data' / 'ds004752'

GATE0_RUN = DRIVE_ROOT / 'artifacts/gate0/20260417T102811097110Z'
GATE1_RUN = DRIVE_ROOT / 'artifacts/gate1/20260418T153918409528Z'
GATE2_RUN = DRIVE_ROOT / 'artifacts/gate2/20260418T160143330194Z'
PREREG_RUN = DRIVE_ROOT / 'artifacts/prereg/20260418T161442014597Z'
PHASE05_PREFLIGHT_RUN = DRIVE_ROOT / 'artifacts/phase05/20260418T163438037205Z'

PREREG_BUNDLE = PREREG_RUN / 'prereg_bundle.json'
PHASE05_OUTPUT_ROOT = DRIVE_ROOT / 'artifacts/phase05_estimators'

required_paths = {
    'repo': REPO_DIR,
    'dataset_root': DATASET_ROOT,
    'gate0_run': GATE0_RUN,
    'gate1_run': GATE1_RUN,
    'gate2_run': GATE2_RUN,
    'prereg_run': PREREG_RUN,
    'phase05_preflight_run': PHASE05_PREFLIGHT_RUN,
    'prereg_bundle': PREREG_BUNDLE,
}

print('================ Required Paths ================')
for name, path in required_paths.items():
    print(name, path, 'exists =', path.exists())
    if not path.exists():
        raise FileNotFoundError(f'Missing required path: {name} -> {path}')

print('Phase 0.5 estimator output root:', PHASE05_OUTPUT_ROOT)


In [ ]:
# ============================================================
# Cell 3: Verify prereg bundle, Phase 0.5 preflight, hash chain
# ============================================================
# Mục tiêu:
# - Kiểm tra prereg bundle locked.
# - Kiểm tra Phase 0.5 preflight bám đúng prereg hash.
# - Kiểm tra bundle trỏ đúng Gate 0/1/2 source of truth.
# ============================================================

import json
from pathlib import Path

bundle = json.loads(PREREG_BUNDLE.read_text())
phase05_summary = json.loads((PHASE05_PREFLIGHT_RUN / 'phase05_summary.json').read_text())
phase05_inputs = json.loads((PHASE05_PREFLIGHT_RUN / 'phase05_inputs.json').read_text())

print('================ Source Integrity ================')
print('Bundle status:', bundle['status'])
print('Bundle hash:', bundle['prereg_bundle_hash_sha256'])
print('Phase 0.5 status:', phase05_summary['status'])
print('Phase 0.5 hash:', phase05_summary['prereg_bundle_hash_sha256'])
print('Gate 0 in bundle:', bundle['source_runs']['gate0'])
print('Gate 1 in bundle:', bundle['source_runs']['gate1'])
print('Gate 2 in bundle:', bundle['source_runs']['gate2'])

assert bundle['status'] == 'locked'
assert Path(bundle['source_runs']['gate0']) == GATE0_RUN
assert Path(bundle['source_runs']['gate1']) == GATE1_RUN
assert Path(bundle['source_runs']['gate2']) == GATE2_RUN
assert phase05_summary['status'] == 'phase05_observability_preflight_ready'
assert phase05_summary['prereg_bundle_hash_sha256'] == bundle['prereg_bundle_hash_sha256']
assert phase05_inputs['prereg_bundle_hash_sha256'] == bundle['prereg_bundle_hash_sha256']

print('OK: Source-of-truth chain is consistent.')


In [ ]:
# ============================================================
# Cell 4: Ghi rõ execution rules dùng cho notebook này
# ============================================================
# Các rule này được đối chiếu từ Technical Spec / Annex / Dossier:
# - Phase 0.5 chỉ observability-only.
# - Mọi fit observability phải nằm trong outer-train.
# - Teacher observable cần task contrast và controls.
# - Spatial/ICA controls nếu chưa chạy thì phải là blocker, không được giả lập thành pass.
# ============================================================

execution_rules = {
    'phase': 'phase05_observability_estimators',
    'decoder_training_allowed': False,
    'privileged_transfer_claim_allowed': False,
    'outer_test_fit_allowed': False,
    'required_observability_quantities': ['Q2_task', 'Q2_base', 'DeltaQ2_obs', 'p_perm'],
    'required_controls': [
        'task_vs_matched_control',
        'grouped_teacher_permutation',
        'nuisance_only_control',
        'spatial_permutation_control',
        'ica_robustness_control',
    ],
    'integrity_policy': 'If a required control is not implemented, report it as blocker; do not mark teacher survival final.',
}

print(json.dumps(execution_rules, indent=2, ensure_ascii=False))


In [ ]:
# ============================================================
# Cell 5: Run tests and verify CLI availability
# ============================================================
# Nếu tests fail thì dừng. Không chạy estimator trên dữ liệu thật khi code chưa qua test.
# ============================================================

import subprocess

print('\n$ python -m unittest discover -s tests')
subprocess.run(['python', '-m', 'unittest', 'discover', '-s', 'tests'], cwd=REPO_DIR, check=True)

print('\n$ python -m src.cli phase05_estimators --help')
subprocess.run(['python', '-m', 'src.cli', 'phase05_estimators', '--help'], cwd=REPO_DIR, check=True)


In [ ]:
# ============================================================
# Cell 6: Data materialization sanity check
# ============================================================
# Kiểm tra một vài EDF/iEEG payload đã materialized thật.
# Không tải lại dữ liệu ở đây; nếu thiếu payload thì quay lại DataLad materialization.
# ============================================================

sample_paths = [
    DATASET_ROOT / 'sub-01/ses-01/eeg/sub-01_ses-01_task-verbalWM_run-01_eeg.edf',
    DATASET_ROOT / 'sub-01/ses-01/ieeg/sub-01_ses-01_task-verbalWM_run-01_ieeg.edf',
    DATASET_ROOT / 'sub-15/ses-04/eeg/sub-15_ses-04_task-verbalWM_run-01_eeg.edf',
    DATASET_ROOT / 'sub-15/ses-04/ieeg/sub-15_ses-04_task-verbalWM_run-01_ieeg.edf',
]

print('================ Payload Sanity Check ================')
for path in sample_paths:
    target = path.resolve()
    exists = target.exists()
    size_mb = round(target.stat().st_size / 1024 / 1024, 2) if exists else None
    print('link:', path)
    print('  target_exists:', exists)
    print('  target_size_mb:', size_mb)
    if not exists:
        raise FileNotFoundError(f'Missing materialized payload target: {path}')


In [ ]:
# ============================================================
# Cell 7: Run Phase 0.5 estimator smoke run
# ============================================================
# Smoke run mặc định:
# - 3 subjects đầu tiên.
# - 6 sessions đầu tiên.
# - 24 clean trials/session.
# - 20 grouped permutations.
#
# Đây là implementation/compute smoke, không phải final inference.
# Final inference tối thiểu phải tăng permutations theo registry/config và chạy full cohort.
# ============================================================

SMOKE_MAX_SUBJECTS = 3
SMOKE_MAX_SESSIONS = 6
SMOKE_MAX_TRIALS_PER_SESSION = 24
SMOKE_N_PERMUTATIONS = 20

cmd = [
    'python', '-m', 'src.cli', 'phase05_estimators',
    '--profile', 't4_safe',
    '--prereg-bundle', str(PREREG_BUNDLE),
    '--phase05-run', str(PHASE05_PREFLIGHT_RUN),
    '--dataset-root', str(DATASET_ROOT),
    '--config', 'configs/phase05/estimators.json',
    '--output-root', str(PHASE05_OUTPUT_ROOT),
    '--max-subjects', str(SMOKE_MAX_SUBJECTS),
    '--max-sessions', str(SMOKE_MAX_SESSIONS),
    '--max-trials-per-session', str(SMOKE_MAX_TRIALS_PER_SESSION),
    '--n-permutations', str(SMOKE_N_PERMUTATIONS),
]

print('\n$', ' '.join(cmd))
subprocess.run(cmd, cwd=REPO_DIR, check=True)


In [ ]:
# ============================================================
# Cell 8: Inspect smoke run artifacts
# ============================================================
# Mục tiêu:
# - Kiểm tra artifact bắt buộc.
# - In summary ngắn.
# - Xác nhận claim_ready = False.
# ============================================================

LATEST_ESTIMATORS = PHASE05_OUTPUT_ROOT / 'latest.txt'
if not LATEST_ESTIMATORS.exists():
    raise FileNotFoundError(f'Missing latest estimator pointer: {LATEST_ESTIMATORS}')

estimator_run = Path(LATEST_ESTIMATORS.read_text().strip())
summary_path = estimator_run / 'phase05_estimators_summary.json'
summary = json.loads(summary_path.read_text())

required_files = [
    'phase05_estimator_inputs.json',
    'feature_extraction_report.json',
    'task_contrast_observability_results.json',
    'controls_report.json',
    'teacher_survival_table.json',
    'coverage_registry.json',
    'phase05_estimators_report.md',
    'phase05_estimators_summary.json',
]

print('================ Phase 0.5 Estimator Smoke Summary ================')
print('Run dir:', estimator_run)
print('Status:', summary['status'])
print('Subjects:', summary['n_subjects'])
print('Sessions:', summary['n_sessions'])
print('Trials:', summary['n_trials'])
print('Teacher targets:', summary['n_teacher_targets'])
print('Permutations:', summary['n_permutations'])
print('Permutation inference:', summary['permutation_inference_status'])
print('Claim ready:', summary['claim_ready'])
print('Controls blockers:', summary['controls_blockers'])
print('Next step:', summary['next_step'])

assert summary['does_not_train_decoder'] is True
assert summary['does_not_estimate_privileged_transfer_efficacy'] is True
assert summary['claim_ready'] is False

print('\n================ Artifact Check ================')
for name in required_files:
    path = estimator_run / name
    print(name, 'exists =', path.exists())
    if not path.exists():
        raise FileNotFoundError(f'Missing estimator artifact: {path}')

print('\nOK: Smoke estimator run completed with explicit non-claim status.')


In [ ]:
# ============================================================
# Cell 9: Optional full-cohort estimator run
# ============================================================
# Mặc định tắt để tránh chạy lâu ngoài ý muốn.
# Chỉ bật RUN_FULL_ESTIMATORS = True khi:
# - Drive còn đủ dung lượng.
# - Colab runtime ổn định.
# - Bạn chấp nhận thời gian chạy dài hơn.
#
# Dù full run xong, nếu spatial/ICA controls còn blocker thì vẫn chưa claim-ready.
# ============================================================

RUN_FULL_ESTIMATORS = False

FULL_MAX_SUBJECTS = 15
FULL_MAX_SESSIONS = 68
FULL_MAX_TRIALS_PER_SESSION = 0  # 0 = dùng toàn bộ clean trials được chọn trong CLI.
FULL_N_PERMUTATIONS = 200

if RUN_FULL_ESTIMATORS:
    full_cmd = [
        'python', '-m', 'src.cli', 'phase05_estimators',
        '--profile', 't4_safe',
        '--prereg-bundle', str(PREREG_BUNDLE),
        '--phase05-run', str(PHASE05_PREFLIGHT_RUN),
        '--dataset-root', str(DATASET_ROOT),
        '--config', 'configs/phase05/estimators.json',
        '--output-root', str(PHASE05_OUTPUT_ROOT),
        '--max-subjects', str(FULL_MAX_SUBJECTS),
        '--max-sessions', str(FULL_MAX_SESSIONS),
        '--max-trials-per-session', str(FULL_MAX_TRIALS_PER_SESSION),
        '--n-permutations', str(FULL_N_PERMUTATIONS),
    ]
    print('\n$', ' '.join(full_cmd))
    subprocess.run(full_cmd, cwd=REPO_DIR, check=True)
else:
    print('Full estimator run is disabled. Set RUN_FULL_ESTIMATORS = True to run full cohort.')


In [ ]:
# ============================================================
# Cell 10: Register estimator source of truth pointer
# ============================================================
# Ghi lại run hiện tại để bước sau không dùng latest.txt mơ hồ.
# Nếu bạn đã chạy full run ở Cell 9, pointer sẽ trỏ full run mới nhất.
# Nếu chưa, pointer trỏ smoke run và phải được ghi rõ là smoke/non-claim.
# ============================================================

from datetime import datetime, timezone

estimator_run = Path((PHASE05_OUTPUT_ROOT / 'latest.txt').read_text().strip())
summary = json.loads((estimator_run / 'phase05_estimators_summary.json').read_text())
controls = json.loads((estimator_run / 'controls_report.json').read_text())

source_record = {
    'status': 'phase05_estimators_source_recorded',
    'created_utc': datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'),
    'estimator_run': str(estimator_run),
    'summary_path': str(estimator_run / 'phase05_estimators_summary.json'),
    'phase05_preflight_source_of_truth': str(PHASE05_PREFLIGHT_RUN),
    'prereg_bundle': str(PREREG_BUNDLE),
    'prereg_bundle_hash_sha256': bundle['prereg_bundle_hash_sha256'],
    'run_status': summary['status'],
    'claim_ready': summary['claim_ready'],
    'controls_blockers': controls['blockers'],
    'scientific_integrity_limits': summary['scientific_integrity_limits'],
}

source_path = estimator_run / 'phase05_estimators_source_of_truth.json'
source_path.write_text(json.dumps(source_record, indent=2, ensure_ascii=False) + '\n')

print('================ Source Record ================')
print('Written:', source_path)
print('Estimator source:', estimator_run)
print('Run status:', summary['status'])
print('Claim ready:', summary['claim_ready'])
print('Controls blockers:', controls['blockers'])


In [ ]:
# ============================================================
# Cell 11: Final interpretation
# ============================================================
# Cell này không thêm tính toán. Chỉ in diễn giải đúng giới hạn.
# ============================================================

report_text = (estimator_run / 'phase05_estimators_report.md').read_text()
print('================ Report Preview ================')
print(report_text[:3000])

print('\n================ Final Interpretation ================')
print('OK: Phase 0.5 estimator workflow ran under the locked prereg bundle and fixed Phase 0.5 preflight source.')
print('OK: Feature extraction, task/base contrast, grouped teacher permutation, and nuisance-only control outputs were written.')
print('NOT OK TO CLAIM: This is not decoder training and not privileged-transfer efficacy evidence.')
print('NOT FINAL IF BLOCKERS EXIST: Teacher survival remains draft while spatial/ICA controls or final permutation count are incomplete.')
print('NEXT: Implement spatial permutation and ICA robustness controls, then rerun with final permutation count before any teacher-survival claim.')
print('Phase 0.5 estimator source of truth:', estimator_run)


In [ ]:
# ============================================================
# Cell 12: Pull spatial-control update and verify code
# ============================================================
# Mục tiêu:
# - Pull commit mới có rowwise spatial permutation control.
# - Chạy tests trước khi rerun estimator.
# - Không chạy decoder, không tạo claim efficacy.
# ============================================================

from pathlib import Path
import subprocess
import json

# Fallback nếu chạy cell này độc lập sau khi mở lại runtime.
REPO_DIR = Path(globals().get('REPO_DIR', '/content/eeg-ds004752'))
DRIVE_ROOT = Path(globals().get('DRIVE_ROOT', '/content/drive/MyDrive/eeg-ds004752'))
DATASET_ROOT = Path(globals().get('DATASET_ROOT', DRIVE_ROOT / 'data' / 'ds004752'))
PREREG_RUN = Path(globals().get('PREREG_RUN', DRIVE_ROOT / 'artifacts/prereg/20260418T161442014597Z'))
PHASE05_PREFLIGHT_RUN = Path(globals().get('PHASE05_PREFLIGHT_RUN', DRIVE_ROOT / 'artifacts/phase05/20260418T163438037205Z'))
PREREG_BUNDLE = Path(globals().get('PREREG_BUNDLE', PREREG_RUN / 'prereg_bundle.json'))
PHASE05_OUTPUT_ROOT = Path(globals().get('PHASE05_OUTPUT_ROOT', DRIVE_ROOT / 'artifacts/phase05_estimators'))

if not REPO_DIR.exists():
    raise FileNotFoundError(f'Missing repo: {REPO_DIR}. Chạy lại Cell 1 trước.')

%cd /content/eeg-ds004752

print('\n$ git pull')
subprocess.run(['git', 'pull'], cwd=REPO_DIR, check=True)

print('\n$ git log --oneline -5')
subprocess.run(['git', 'log', '--oneline', '-5'], cwd=REPO_DIR, check=True)

print('\n$ python -m unittest discover -s tests')
subprocess.run(['python', '-m', 'unittest', 'discover', '-s', 'tests'], cwd=REPO_DIR, check=True)

print('\n$ python -m src.cli phase05_estimators --help')
subprocess.run(['python', '-m', 'src.cli', 'phase05_estimators', '--help'], cwd=REPO_DIR, check=True)


In [ ]:
# ============================================================
# Cell 13: Rerun Phase 0.5 estimator smoke with spatial control
# ============================================================
# Mục tiêu:
# - Rerun smoke estimator bằng code mới.
# - Spatial permutation control giờ được tính thật dưới dạng rowwise channel shuffle.
# - ICA robustness vẫn là blocker nếu chưa có ICA branch feature engine.
#
# Smoke run vẫn KHÔNG phải final inference:
# - Chỉ 3 subjects / 6 sessions.
# - 24 clean trials/session.
# - 20 permutations < final minimum 200.
# ============================================================

SPATIAL_SMOKE_MAX_SUBJECTS = 3
SPATIAL_SMOKE_MAX_SESSIONS = 6
SPATIAL_SMOKE_MAX_TRIALS_PER_SESSION = 24
SPATIAL_SMOKE_N_PERMUTATIONS = 20

cmd = [
    'python', '-m', 'src.cli', 'phase05_estimators',
    '--profile', 't4_safe',
    '--prereg-bundle', str(PREREG_BUNDLE),
    '--phase05-run', str(PHASE05_PREFLIGHT_RUN),
    '--dataset-root', str(DATASET_ROOT),
    '--config', 'configs/phase05/estimators.json',
    '--output-root', str(PHASE05_OUTPUT_ROOT),
    '--max-subjects', str(SPATIAL_SMOKE_MAX_SUBJECTS),
    '--max-sessions', str(SPATIAL_SMOKE_MAX_SESSIONS),
    '--max-trials-per-session', str(SPATIAL_SMOKE_MAX_TRIALS_PER_SESSION),
    '--n-permutations', str(SPATIAL_SMOKE_N_PERMUTATIONS),
]

print('\n$', ' '.join(cmd))
subprocess.run(cmd, cwd=REPO_DIR, check=True)


In [ ]:
# ============================================================
# Cell 14: Inspect spatial-control run and blockers
# ============================================================
# Mục tiêu:
# - Đọc output mới nhất.
# - Xác nhận q2_spatial đã xuất hiện trong fold results.
# - Xác nhận spatial_permutation_control_not_computed không còn là blocker.
# - Xác nhận ICA và permutation-count vẫn được báo cáo trung thực nếu còn thiếu.
# ============================================================

LATEST_ESTIMATORS = PHASE05_OUTPUT_ROOT / 'latest.txt'
if not LATEST_ESTIMATORS.exists():
    raise FileNotFoundError(f'Missing latest estimator pointer: {LATEST_ESTIMATORS}')

spatial_run = Path(LATEST_ESTIMATORS.read_text().strip())
summary = json.loads((spatial_run / 'phase05_estimators_summary.json').read_text())
controls = json.loads((spatial_run / 'controls_report.json').read_text())
observability = json.loads((spatial_run / 'task_contrast_observability_results.json').read_text())
teacher_survival = json.loads((spatial_run / 'teacher_survival_table.json').read_text())

first_teacher = observability['fold_results'][0]
first_fold = first_teacher['fold_results'][0]

print('================ Spatial-Control Estimator Summary ================')
print('Run dir:', spatial_run)
print('Status:', summary['status'])
print('Subjects:', summary['n_subjects'])
print('Sessions:', summary['n_sessions'])
print('Trials:', summary['n_trials'])
print('Teacher targets:', summary['n_teacher_targets'])
print('Permutations:', summary['n_permutations'])
print('Permutation inference:', summary['permutation_inference_status'])
print('Claim ready:', summary['claim_ready'])
print('Controls status:', controls['status'])
print('Implemented controls:', controls['implemented_controls'])
print('Pending controls:', controls['pending_controls'])
print('Blockers:', controls['blockers'])

print('\n================ First Fold Example ================')
print('Teacher:', first_teacher['teacher_id'])
print(json.dumps(first_fold, indent=2, ensure_ascii=False)[:3000])

assert 'q2_spatial' in first_fold
assert 'spatial_veto' in first_fold
assert 'rowwise_spatial_permutation_control' in controls['implemented_controls']
assert 'spatial_permutation_control_not_computed' not in controls['blockers']
assert 'ica_robustness_control_not_computed' in controls['blockers']
assert summary['claim_ready'] is False
assert summary['does_not_train_decoder'] is True
assert summary['does_not_estimate_privileged_transfer_efficacy'] is True

print('\n================ Interpretation ================')
print('OK: Spatial permutation control is now computed and recorded as q2_spatial/spatial_veto.')
print('OK: Spatial-not-computed blocker is cleared for this run.')
print('NOT FINAL: ICA robustness remains a blocker.')
print('NOT FINAL: 20 permutations is smoke-only, below final minimum 200.')
print('NOT OK TO CLAIM: No decoder training or privileged-transfer efficacy evidence was produced.')
print('Spatial-control estimator source of truth:', spatial_run)


In [ ]:
# ============================================================
# Cell 15: Optional final-permutation rerun with spatial control
# ============================================================
# Mục tiêu:
# - Rerun với n_permutations = 200 để xóa blocker permutation_count_below_final_minimum.
# - Vẫn KHÔNG xóa ICA blocker nếu ICA branch chưa được triển khai.
# - Chưa train decoder và chưa claim privileged-transfer efficacy.
#
# Khuyến nghị:
# - Chạy trên T4 nếu full cohort chậm.
# - Có thể bắt đầu với subset rồi mới tăng FULL_MAX_SUBJECTS/FULL_MAX_SESSIONS.
# ============================================================

RUN_SPATIAL_FINAL_PERMUTATION = False

FINAL_MAX_SUBJECTS = 15
FINAL_MAX_SESSIONS = 68
FINAL_MAX_TRIALS_PER_SESSION = 0  # 0 = dùng toàn bộ clean trials được CLI chọn.
FINAL_N_PERMUTATIONS = 200

if RUN_SPATIAL_FINAL_PERMUTATION:
    final_cmd = [
        'python', '-m', 'src.cli', 'phase05_estimators',
        '--profile', 't4_safe',
        '--prereg-bundle', str(PREREG_BUNDLE),
        '--phase05-run', str(PHASE05_PREFLIGHT_RUN),
        '--dataset-root', str(DATASET_ROOT),
        '--config', 'configs/phase05/estimators.json',
        '--output-root', str(PHASE05_OUTPUT_ROOT),
        '--max-subjects', str(FINAL_MAX_SUBJECTS),
        '--max-sessions', str(FINAL_MAX_SESSIONS),
        '--max-trials-per-session', str(FINAL_MAX_TRIALS_PER_SESSION),
        '--n-permutations', str(FINAL_N_PERMUTATIONS),
    ]
    print('\n$', ' '.join(final_cmd))
    subprocess.run(final_cmd, cwd=REPO_DIR, check=True)

    final_run = Path((PHASE05_OUTPUT_ROOT / 'latest.txt').read_text().strip())
    final_summary = json.loads((final_run / 'phase05_estimators_summary.json').read_text())
    final_controls = json.loads((final_run / 'controls_report.json').read_text())

    print('\n================ Final-Permutation Run Summary ================')
    print('Run dir:', final_run)
    print('Status:', final_summary['status'])
    print('Subjects:', final_summary['n_subjects'])
    print('Sessions:', final_summary['n_sessions'])
    print('Trials:', final_summary['n_trials'])
    print('Permutations:', final_summary['n_permutations'])
    print('Permutation inference:', final_summary['permutation_inference_status'])
    print('Blockers:', final_controls['blockers'])
    print('Claim ready:', final_summary['claim_ready'])

    assert 'permutation_count_below_final_minimum' not in final_controls['blockers']
    assert 'ica_robustness_control_not_computed' in final_controls['blockers']
    assert final_summary['claim_ready'] is False
else:
    print('Final-permutation rerun is disabled.')
    print('Set RUN_SPATIAL_FINAL_PERMUTATION = True when you want to run 200 permutations.')
    print('Even after this run, ICA robustness remains a blocker until implemented.')


In [ ]:
# ============================================================
# Cell 16: Pull ICA robustness update and verify code
# ============================================================
# Mục tiêu:
# - Pull commit mới có ICA robustness control.
# - Cài runtime extras gồm mne, scikit-learn, mne-icalabel.
# - Chạy tests trước khi rerun estimator.
#
# Cơ sở tài liệu:
# - ICA là stress-test branch, không phải decoder.
# - ICA robustness yêu cầu o_ica / (o_min + eps) >= 0.7.
# - Nếu ICA/ICLabel không chạy được, phải ghi blocker, không được giả lập pass.
# ============================================================

from pathlib import Path
import subprocess
import json

REPO_DIR = Path(globals().get('REPO_DIR', '/content/eeg-ds004752'))
DRIVE_ROOT = Path(globals().get('DRIVE_ROOT', '/content/drive/MyDrive/eeg-ds004752'))
DATASET_ROOT = Path(globals().get('DATASET_ROOT', DRIVE_ROOT / 'data' / 'ds004752'))
PREREG_RUN = Path(globals().get('PREREG_RUN', DRIVE_ROOT / 'artifacts/prereg/20260418T161442014597Z'))
PHASE05_PREFLIGHT_RUN = Path(globals().get('PHASE05_PREFLIGHT_RUN', DRIVE_ROOT / 'artifacts/phase05/20260418T163438037205Z'))
PREREG_BUNDLE = Path(globals().get('PREREG_BUNDLE', PREREG_RUN / 'prereg_bundle.json'))
PHASE05_OUTPUT_ROOT = Path(globals().get('PHASE05_OUTPUT_ROOT', DRIVE_ROOT / 'artifacts/phase05_estimators'))

if not REPO_DIR.exists():
    raise FileNotFoundError(f'Missing repo: {REPO_DIR}. Chạy lại Cell 1 trước.')

%cd /content/eeg-ds004752

print('\n$ git pull')
subprocess.run(['git', 'pull'], cwd=REPO_DIR, check=True)

print('\n$ INSTALL_SIGNAL_EXTRAS=1 bash bootstrap/install_runtime.sh')
subprocess.run(['bash', '-lc', 'INSTALL_SIGNAL_EXTRAS=1 bash bootstrap/install_runtime.sh'], cwd=REPO_DIR, check=True)

print('\n$ git log --oneline -5')
subprocess.run(['git', 'log', '--oneline', '-5'], cwd=REPO_DIR, check=True)

print('\n$ python -m unittest discover -s tests')
subprocess.run(['python', '-m', 'unittest', 'discover', '-s', 'tests'], cwd=REPO_DIR, check=True)

print('\n$ python -m src.cli phase05_estimators --help')
subprocess.run(['python', '-m', 'src.cli', 'phase05_estimators', '--help'], cwd=REPO_DIR, check=True)


In [ ]:
# ============================================================
# Cell 17: Rerun Phase 0.5 estimator smoke with spatial + ICA
# ============================================================
# Mục tiêu:
# - Rerun smoke estimator bằng code mới.
# - Spatial permutation control đã có từ bước trước.
# - ICA robustness control sẽ fit ICA stress-test branch trên outer-train và áp cho fold.
#
# Smoke run vẫn KHÔNG phải final inference:
# - 3 subjects / 6 sessions.
# - 24 clean trials/session.
# - 20 permutations < final minimum 200.
# ============================================================

ICA_SMOKE_MAX_SUBJECTS = 3
ICA_SMOKE_MAX_SESSIONS = 6
ICA_SMOKE_MAX_TRIALS_PER_SESSION = 24
ICA_SMOKE_N_PERMUTATIONS = 20

cmd = [
    'python', '-m', 'src.cli', 'phase05_estimators',
    '--profile', 't4_safe',
    '--prereg-bundle', str(PREREG_BUNDLE),
    '--phase05-run', str(PHASE05_PREFLIGHT_RUN),
    '--dataset-root', str(DATASET_ROOT),
    '--config', 'configs/phase05/estimators.json',
    '--output-root', str(PHASE05_OUTPUT_ROOT),
    '--max-subjects', str(ICA_SMOKE_MAX_SUBJECTS),
    '--max-sessions', str(ICA_SMOKE_MAX_SESSIONS),
    '--max-trials-per-session', str(ICA_SMOKE_MAX_TRIALS_PER_SESSION),
    '--n-permutations', str(ICA_SMOKE_N_PERMUTATIONS),
]

print('\n$', ' '.join(cmd))
subprocess.run(cmd, cwd=REPO_DIR, check=True)


In [ ]:
# ============================================================
# Cell 18: Inspect ICA robustness smoke run
# ============================================================
# Mục tiêu:
# - Xác nhận q2_ica, ica_ratio, ica_veto đã xuất hiện trong fold results.
# - Xác nhận ICA blocker chỉ được xóa nếu ica_control_status = computed.
# - Xác nhận vẫn chưa final nếu permutation count < 200.
# ============================================================

LATEST_ESTIMATORS = PHASE05_OUTPUT_ROOT / 'latest.txt'
if not LATEST_ESTIMATORS.exists():
    raise FileNotFoundError(f'Missing latest estimator pointer: {LATEST_ESTIMATORS}')

ica_run = Path(LATEST_ESTIMATORS.read_text().strip())
summary = json.loads((ica_run / 'phase05_estimators_summary.json').read_text())
controls = json.loads((ica_run / 'controls_report.json').read_text())
observability = json.loads((ica_run / 'task_contrast_observability_results.json').read_text())
teacher_survival = json.loads((ica_run / 'teacher_survival_table.json').read_text())

first_teacher = observability['fold_results'][0]
first_fold = first_teacher['fold_results'][0]

print('================ ICA Robustness Smoke Summary ================')
print('Run dir:', ica_run)
print('Status:', summary['status'])
print('Subjects:', summary['n_subjects'])
print('Sessions:', summary['n_sessions'])
print('Trials:', summary['n_trials'])
print('Teacher targets:', summary['n_teacher_targets'])
print('Permutations:', summary['n_permutations'])
print('Permutation inference:', summary['permutation_inference_status'])
print('Claim ready:', summary['claim_ready'])
print('Phase 0.5 observability table ready:', summary.get('phase05_observability_table_ready'))
print('Controls status:', controls['status'])
print('ICA control status:', controls.get('ica_control_status'))
print('Implemented controls:', controls['implemented_controls'])
print('Pending controls:', controls['pending_controls'])
print('Blockers:', controls['blockers'])

print('\n================ ICA Diagnostics ================')
print(json.dumps(controls.get('ica_diagnostics'), indent=2, ensure_ascii=False)[:5000])

print('\n================ First Fold Example ================')
print('Teacher:', first_teacher['teacher_id'])
print(json.dumps(first_fold, indent=2, ensure_ascii=False)[:3000])

assert 'q2_spatial' in first_fold
assert 'spatial_veto' in first_fold
assert 'q2_ica' in first_fold
assert 'ica_ratio' in first_fold
assert 'ica_veto' in first_fold
assert 'ica_robustness_control' in controls['implemented_controls']
assert summary['claim_ready'] is False
assert summary['does_not_train_decoder'] is True
assert summary['does_not_estimate_privileged_transfer_efficacy'] is True

if controls.get('ica_control_status') == 'computed':
    assert 'ica_robustness_control_not_computed' not in controls['blockers']
    print('\nOK: ICA robustness control computed; ICA-not-computed blocker is cleared.')
else:
    assert 'ica_robustness_control_not_computed' in controls['blockers']
    print('\nNOT FINAL: ICA robustness did not compute successfully. See ICA diagnostics above.')

assert 'permutation_count_below_final_minimum' in controls['blockers']

print('\n================ Interpretation ================')
print('OK: ICA smoke inspection completed without claiming decoder efficacy.')
print('NOT FINAL: 20 permutations is smoke-only, below final minimum 200.')
print('NOT OK TO CLAIM: No Phase 1 decoder was trained and no privileged-transfer efficacy evidence was produced.')
print('ICA smoke source of truth:', ica_run)


In [ ]:
# ============================================================
# Cell 19: Final 200-permutation run after spatial + ICA smoke
# ============================================================
# Mục tiêu:
# - Chỉ chạy sau khi Cell 18 cho thấy ICA control status = computed.
# - Rerun với n_permutations = 200 để xóa blocker permutation_count_below_final_minimum.
# - Vẫn không train decoder và không claim privileged-transfer efficacy.
#
# Khuyến nghị phần cứng:
# - T4 là hợp lý.
# - Nếu full cohort quá lâu, giữ subset trước rồi mới tăng dần.
# ============================================================

RUN_ICA_FINAL_200_PERMUTATIONS = False

FINAL_MAX_SUBJECTS = 15
FINAL_MAX_SESSIONS = 68
FINAL_MAX_TRIALS_PER_SESSION = 0  # 0 = dùng toàn bộ clean trials được CLI chọn.
FINAL_N_PERMUTATIONS = 200

if RUN_ICA_FINAL_200_PERMUTATIONS:
    final_cmd = [
        'python', '-m', 'src.cli', 'phase05_estimators',
        '--profile', 't4_safe',
        '--prereg-bundle', str(PREREG_BUNDLE),
        '--phase05-run', str(PHASE05_PREFLIGHT_RUN),
        '--dataset-root', str(DATASET_ROOT),
        '--config', 'configs/phase05/estimators.json',
        '--output-root', str(PHASE05_OUTPUT_ROOT),
        '--max-subjects', str(FINAL_MAX_SUBJECTS),
        '--max-sessions', str(FINAL_MAX_SESSIONS),
        '--max-trials-per-session', str(FINAL_MAX_TRIALS_PER_SESSION),
        '--n-permutations', str(FINAL_N_PERMUTATIONS),
    ]
    print('\n$', ' '.join(final_cmd))
    subprocess.run(final_cmd, cwd=REPO_DIR, check=True)

    final_run = Path((PHASE05_OUTPUT_ROOT / 'latest.txt').read_text().strip())
    final_summary = json.loads((final_run / 'phase05_estimators_summary.json').read_text())
    final_controls = json.loads((final_run / 'controls_report.json').read_text())
    final_survival = json.loads((final_run / 'teacher_survival_table.json').read_text())

    print('\n================ Final 200-Permutation Summary ================')
    print('Run dir:', final_run)
    print('Status:', final_summary['status'])
    print('Subjects:', final_summary['n_subjects'])
    print('Sessions:', final_summary['n_sessions'])
    print('Trials:', final_summary['n_trials'])
    print('Teacher targets:', final_summary['n_teacher_targets'])
    print('Permutations:', final_summary['n_permutations'])
    print('Permutation inference:', final_summary['permutation_inference_status'])
    print('Controls status:', final_controls['status'])
    print('ICA control status:', final_controls.get('ica_control_status'))
    print('Blockers:', final_controls['blockers'])
    print('Teacher survival status:', final_survival['status'])
    print('Phase 0.5 observability table ready:', final_summary.get('phase05_observability_table_ready'))
    print('Claim ready for privileged-transfer efficacy:', final_summary['claim_ready'])

    assert final_summary['n_permutations'] >= 200
    assert 'permutation_count_below_final_minimum' not in final_controls['blockers']
    assert final_summary['claim_ready'] is False
    assert final_summary['does_not_train_decoder'] is True
    assert final_summary['does_not_estimate_privileged_transfer_efficacy'] is True

    if final_controls.get('ica_control_status') == 'computed':
        assert 'ica_robustness_control_not_computed' not in final_controls['blockers']
        print('\nOK: Spatial + ICA + final permutation count are complete for Phase 0.5 observability tables.')
    else:
        print('\nNOT FINAL: ICA is still incomplete; do not use this as final Phase 0.5 table.')

    print('Final Phase 0.5 estimator source of truth:', final_run)
else:
    print('Final 200-permutation rerun is disabled.')
    print('Run Cell 18 first. Only set RUN_ICA_FINAL_200_PERMUTATIONS = True if ICA control status is computed.')


In [ ]:
# ============================================================
# Cell 20: Pull EDF timestamp fallback update
# ============================================================
# Mục tiêu:
# - Pull code mới xử lý lỗi EDF header: "second must be in 0..59".
# - Fallback chỉ patch bản sao tạm của EDF header starttime seconds về 59.
# - Signal samples không bị sửa trong dataset gốc.
# - Nếu fallback vẫn fail, workflow sẽ ghi phase05_estimator_exclusion_note.json.
# ============================================================

from pathlib import Path
import subprocess
import json

REPO_DIR = Path(globals().get('REPO_DIR', '/content/eeg-ds004752'))
DRIVE_ROOT = Path(globals().get('DRIVE_ROOT', '/content/drive/MyDrive/eeg-ds004752'))
DATASET_ROOT = Path(globals().get('DATASET_ROOT', DRIVE_ROOT / 'data' / 'ds004752'))
PREREG_RUN = Path(globals().get('PREREG_RUN', DRIVE_ROOT / 'artifacts/prereg/20260418T161442014597Z'))
PHASE05_PREFLIGHT_RUN = Path(globals().get('PHASE05_PREFLIGHT_RUN', DRIVE_ROOT / 'artifacts/phase05/20260418T163438037205Z'))
PREREG_BUNDLE = Path(globals().get('PREREG_BUNDLE', PREREG_RUN / 'prereg_bundle.json'))
PHASE05_OUTPUT_ROOT = Path(globals().get('PHASE05_OUTPUT_ROOT', DRIVE_ROOT / 'artifacts/phase05_estimators'))

if not REPO_DIR.exists():
    raise FileNotFoundError(f'Missing repo: {REPO_DIR}. Chạy lại Cell 1 trước.')

%cd /content/eeg-ds004752

print('\n$ git pull')
subprocess.run(['git', 'pull'], cwd=REPO_DIR, check=True)

print('\n$ INSTALL_SIGNAL_EXTRAS=1 bash bootstrap/install_runtime.sh')
subprocess.run(['bash', '-lc', 'INSTALL_SIGNAL_EXTRAS=1 bash bootstrap/install_runtime.sh'], cwd=REPO_DIR, check=True)

print('\n$ python -m unittest discover -s tests')
subprocess.run(['python', '-m', 'unittest', 'discover', '-s', 'tests'], cwd=REPO_DIR, check=True)


In [ ]:
# ============================================================
# Cell 21: Rerun full Phase 0.5 estimator with EDF fallback
# ============================================================
# Mục tiêu:
# - Rerun full cohort 15 subjects / 68 sessions / 200 permutations.
# - Kiểm tra fallback có đưa sub-06 ses-04 và sub-14 ses-06 vào estimator hay không.
# - Không train decoder và không claim privileged-transfer efficacy.
# ============================================================

cmd = [
    'python', '-m', 'src.cli', 'phase05_estimators',
    '--profile', 't4_safe',
    '--prereg-bundle', str(PREREG_BUNDLE),
    '--phase05-run', str(PHASE05_PREFLIGHT_RUN),
    '--dataset-root', str(DATASET_ROOT),
    '--config', 'configs/phase05/estimators.json',
    '--output-root', str(PHASE05_OUTPUT_ROOT),
    '--max-subjects', '15',
    '--max-sessions', '68',
    '--max-trials-per-session', '0',
    '--n-permutations', '200',
]

print('\n$', ' '.join(cmd))
subprocess.run(cmd, cwd=REPO_DIR, check=True)


In [ ]:
# ============================================================
# Cell 22: Final closeout audit for Phase 0.5 estimator
# ============================================================
# Mục tiêu:
# - Kiểm tra included/excluded sessions sau EDF fallback.
# - Kiểm tra read_fallbacks được ghi lại.
# - Chỉ đóng Phase 0.5 nếu 68/68 included hoặc exclusion note rõ ràng.
# ============================================================

FINAL_PHASE05_ESTIMATOR_RUN = Path((PHASE05_OUTPUT_ROOT / 'latest.txt').read_text().strip())
summary = json.loads((FINAL_PHASE05_ESTIMATOR_RUN / 'phase05_estimators_summary.json').read_text())
feature_report = json.loads((FINAL_PHASE05_ESTIMATOR_RUN / 'feature_extraction_report.json').read_text())
controls = json.loads((FINAL_PHASE05_ESTIMATOR_RUN / 'controls_report.json').read_text())
exclusion_note = json.loads((FINAL_PHASE05_ESTIMATOR_RUN / 'phase05_estimator_exclusion_note.json').read_text())

print('================ Phase 0.5 Estimator Final Closeout ================')
print('Run:', FINAL_PHASE05_ESTIMATOR_RUN)
print('Status:', summary['status'])
print('Subjects:', summary['n_subjects'])
print('Sessions used:', summary['n_sessions'])
print('Selected sessions:', summary.get('selected_sessions'))
print('Included sessions:', summary.get('included_sessions'))
print('Excluded sessions:', summary.get('excluded_sessions'))
print('Trials:', summary['n_trials'])
print('Teacher targets:', summary['n_teacher_targets'])
print('Permutations:', summary['n_permutations'])
print('Controls blockers:', controls['blockers'])
print('Controls status:', controls['status'])
print('ICA control status:', controls.get('ica_control_status'))
print('Teacher survival status:', summary['teacher_survival_status'])
print('Phase 0.5 table ready:', summary.get('phase05_observability_table_ready'))
print('Claim ready for privileged-transfer efficacy:', summary['claim_ready'])

print('\n================ EDF Read Fallbacks ================')
read_fallbacks = feature_report.get('read_fallbacks') or []
print('Fallback count:', len(read_fallbacks))
print(json.dumps(read_fallbacks, indent=2, ensure_ascii=False)[:5000])

print('\n================ Exclusion Note ================')
print(json.dumps(exclusion_note, indent=2, ensure_ascii=False)[:5000])

assert summary['n_subjects'] == 15
assert summary['n_permutations'] >= 200
assert summary['permutation_inference_status'] == 'final_min_met'
assert controls['blockers'] == []
assert summary['does_not_train_decoder'] is True
assert summary['does_not_estimate_privileged_transfer_efficacy'] is True
assert summary['claim_ready'] is False

if summary.get('included_sessions') == 68 and summary.get('excluded_sessions') == 0:
    print('\nOK: Phase 0.5 estimator includes 68/68 sessions. This can be used as the full-cohort Phase 0.5 estimator source of truth.')
elif exclusion_note.get('status') == 'phase05_estimator_exclusions_recorded':
    print('\nCONDITIONAL OK: Phase 0.5 has explicit exclusion note. Review before closing as exclusion-based source of truth.')
else:
    raise AssertionError('Phase 0.5 closeout is not valid: neither 68/68 included nor explicit exclusion note present.')

print('Final/conditional Phase 0.5 estimator source of truth:', FINAL_PHASE05_ESTIMATOR_RUN)
print('NOT OK TO CLAIM: This still does not prove privileged-transfer efficacy; Phase 1 decoder has not run.')


In [ ]:
# ============================================================
# Cell 23: Lock Phase 0.5 estimator source-of-truth record
# ============================================================
# Mục tiêu:
# - Khóa path/hash của Phase 0.5 estimator full-cohort run.
# - Ghi lại các artifact chính và SHA256 để truy vết.
# - Không chạy thêm estimator.
# - Không tạo claim model efficacy.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import subprocess

REPO_DIR = Path('/content/eeg-ds004752')
PHASE05_ESTIMATOR_SOT = Path('/content/drive/MyDrive/eeg-ds004752/artifacts/phase05_estimators/20260419T130315366518Z')
PHASE05_PREFLIGHT_SOT = Path('/content/drive/MyDrive/eeg-ds004752/artifacts/phase05/20260418T163438037205Z')
PREREG_RUN = Path('/content/drive/MyDrive/eeg-ds004752/artifacts/prereg/20260418T161442014597Z')
PREREG_BUNDLE = PREREG_RUN / 'prereg_bundle.json'

required_files = {
    'phase05_estimators_summary': PHASE05_ESTIMATOR_SOT / 'phase05_estimators_summary.json',
    'feature_extraction_report': PHASE05_ESTIMATOR_SOT / 'feature_extraction_report.json',
    'task_contrast_observability_results': PHASE05_ESTIMATOR_SOT / 'task_contrast_observability_results.json',
    'controls_report': PHASE05_ESTIMATOR_SOT / 'controls_report.json',
    'teacher_survival_table': PHASE05_ESTIMATOR_SOT / 'teacher_survival_table.json',
    'coverage_registry': PHASE05_ESTIMATOR_SOT / 'coverage_registry.json',
    'phase05_estimator_exclusion_note': PHASE05_ESTIMATOR_SOT / 'phase05_estimator_exclusion_note.json',
    'phase05_estimators_report': PHASE05_ESTIMATOR_SOT / 'phase05_estimators_report.md',
}

for name, path in required_files.items():
    print(name, path, 'exists =', path.exists())
    if not path.exists():
        raise FileNotFoundError(f'Missing required artifact: {name} -> {path}')

if not PREREG_BUNDLE.exists():
    raise FileNotFoundError(f'Missing prereg bundle: {PREREG_BUNDLE}')
if not PHASE05_PREFLIGHT_SOT.exists():
    raise FileNotFoundError(f'Missing Phase 0.5 preflight source: {PHASE05_PREFLIGHT_SOT}')


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def git_output(args):
    return subprocess.check_output(args, cwd=REPO_DIR, text=True).strip()


summary = json.loads(required_files['phase05_estimators_summary'].read_text())
controls = json.loads(required_files['controls_report'].read_text())
exclusion_note = json.loads(required_files['phase05_estimator_exclusion_note'].read_text())
bundle = json.loads(PREREG_BUNDLE.read_text())

assert summary['status'] == 'phase05_estimators_controls_complete'
assert summary['n_subjects'] == 15
assert summary['included_sessions'] == 68
assert summary['excluded_sessions'] == 0
assert summary['n_permutations'] >= 200
assert summary['permutation_inference_status'] == 'final_min_met'
assert summary['phase05_observability_table_ready'] is True
assert summary['claim_ready'] is False
assert summary['does_not_train_decoder'] is True
assert summary['does_not_estimate_privileged_transfer_efficacy'] is True
assert controls['blockers'] == []
assert controls['status'] == 'controls_report_complete'
assert controls['ica_control_status'] == 'computed'
assert exclusion_note['status'] == 'no_phase05_estimator_exclusions'
assert exclusion_note['included_sessions'] == 68
assert exclusion_note['excluded_sessions'] == 0
assert summary['prereg_bundle_hash_sha256'] == bundle['prereg_bundle_hash_sha256']

artifact_hashes = {
    name: {'path': str(path), 'sha256': sha256_file(path)}
    for name, path in required_files.items()
}

source_record = {
    'status': 'phase05_estimator_source_of_truth_locked',
    'created_utc': datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'),
    'phase': 'phase05_observability_estimators',
    'source_of_truth_run': str(PHASE05_ESTIMATOR_SOT),
    'phase05_preflight_source_of_truth': str(PHASE05_PREFLIGHT_SOT),
    'prereg_bundle_path': str(PREREG_BUNDLE),
    'prereg_bundle_hash_sha256': bundle['prereg_bundle_hash_sha256'],
    'repo': {
        'path': str(REPO_DIR),
        'branch': git_output(['git', 'rev-parse', '--abbrev-ref', 'HEAD']),
        'commit': git_output(['git', 'rev-parse', 'HEAD']),
        'working_tree_clean': git_output(['git', 'status', '--short']) == '',
        'git_status_short': git_output(['git', 'status', '--short']),
    },
    'validated_summary': {
        'status': summary['status'],
        'subjects': summary['n_subjects'],
        'selected_sessions': summary['selected_sessions'],
        'included_sessions': summary['included_sessions'],
        'excluded_sessions': summary['excluded_sessions'],
        'trials': summary['n_trials'],
        'teacher_targets': summary['n_teacher_targets'],
        'permutations': summary['n_permutations'],
        'permutation_inference_status': summary['permutation_inference_status'],
        'controls_status': controls['status'],
        'controls_blockers': controls['blockers'],
        'ica_control_status': controls['ica_control_status'],
        'teacher_survival_status': summary['teacher_survival_status'],
        'phase05_observability_table_ready': summary['phase05_observability_table_ready'],
        'claim_ready_for_privileged_transfer_efficacy': summary['claim_ready'],
    },
    'edf_read_fallbacks': exclusion_note.get('read_fallbacks', []),
    'artifact_hashes_sha256': artifact_hashes,
    'scientific_integrity_limits': [
        'This source-of-truth record locks Phase 0.5 observability estimator outputs only.',
        'It does not prove decoder performance.',
        'It does not prove privileged-transfer efficacy.',
        'Phase 1 decoder training remains a separate gated phase.',
    ],
}

source_record_path = PHASE05_ESTIMATOR_SOT / 'phase05_estimators_source_of_truth.json'
source_record_path.write_text(json.dumps(source_record, indent=2, ensure_ascii=False) + '\n')

print('\n================ Phase 0.5 Estimator Source Of Truth Locked ================')
print('Status:', source_record['status'])
print('Source of truth:', PHASE05_ESTIMATOR_SOT)
print('Record:', source_record_path)
print('Repo commit:', source_record['repo']['commit'])
print('Working tree clean:', source_record['repo']['working_tree_clean'])
print('Subjects:', source_record['validated_summary']['subjects'])
print('Included sessions:', source_record['validated_summary']['included_sessions'])
print('Excluded sessions:', source_record['validated_summary']['excluded_sessions'])
print('Trials:', source_record['validated_summary']['trials'])
print('Teacher targets:', source_record['validated_summary']['teacher_targets'])
print('Permutations:', source_record['validated_summary']['permutations'])
print('Controls blockers:', source_record['validated_summary']['controls_blockers'])
print('ICA control status:', source_record['validated_summary']['ica_control_status'])
print('Phase 0.5 table ready:', source_record['validated_summary']['phase05_observability_table_ready'])
print('Claim ready for privileged-transfer efficacy:', source_record['validated_summary']['claim_ready_for_privileged_transfer_efficacy'])

print('\nHashes:')
for name, item in artifact_hashes.items():
    print('-', name, item['sha256'])

print('\n================ Interpretation ================')
print('OK: Phase 0.5 estimator source of truth is locked with artifact hashes.')
print('OK: 15 subjects and 68/68 sessions are included; no exclusions remain.')
print('OK: Spatial, nuisance, grouped permutation, and ICA controls are complete for Phase 0.5 observability tables.')
print('NOT OK TO CLAIM: This does not prove decoder performance or privileged-transfer efficacy.')
print('NEXT: Close this notebook and prepare Phase 1 decoder training under the locked source-of-truth chain.')
